# Indoor67 — ResNet18 transfer learning
Compare pretrained ResNet18 with the classical baseline on the **same 4,261 training / 1,066 validation split** and alphabetical mapping of 67 labels. No competition test images are used for model selection.

For CPU efficiency, extract frozen backbone features once, train the new classifier for 10 epochs, then fine-tune `layer4` and the classifier for one epoch. The cached training features contain one seeded augmented view per image; classifier epochs reuse those views. Fine-tuning loads fresh augmentations. Increase training budgets only in a subsequent documented experiment.

The first run downloads official ImageNet weights. Run all cells in the project environment from the repository root or `notebooks/`.

In [ ]:
from pathlib import Path
import copy
import hashlib
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models, transforms
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "dataset.py").is_file()
             and (p / "data" / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.dataset import Indoor67Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = ROOT / "data"
IMAGE_ROOT = DATA_DIR / "images"
CHECKPOINT_DIR = ROOT / "checkpoints"
EXPERIMENT_DIR = ROOT / "experiments"
CHECKPOINT_DIR.mkdir(exist_ok=True)
EXPERIMENT_DIR.mkdir(exist_ok=True)
BEST_PATH = CHECKPOINT_DIR / "resnet18_best.pt"
HISTORY_PATH = EXPERIMENT_DIR / "resnet18_history.csv"
BATCH_SIZE = 32
HEAD_EPOCHS = 10
FINETUNE_EPOCHS = 1
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4
print("Device:", DEVICE, "| CPU threads:", torch.get_num_threads())

## 1. Reuse and verify the saved split
Require the exact current manifest and both partitions to cover every class. Store a fingerprint of the split in the model checkpoint so later comparisons can verify their data provenance.

In [ ]:
manifest = pd.read_csv(DATA_DIR / "train.csv", dtype=str).rename(
    columns={"id": "image_id", "file_path": "image_path"}
)
SPLIT_CSV = DATA_DIR / "validation_split.csv"
if not SPLIT_CSV.is_file():
    raise FileNotFoundError("Run notebook 01 to create validation_split.csv first.")
split_df = pd.read_csv(SPLIT_CSV, dtype=str)
keys = ["image_id", "image_path", "label"]
for name, frame in [("manifest", manifest), ("split", split_df)]:
    if frame.empty or not set(keys).issubset(frame.columns):
        raise ValueError(f"Invalid {name}: expected nonempty {keys} columns.")
    if frame[keys].isna().any().any() or frame.image_id.duplicated().any() or frame.image_path.duplicated().any():
        raise ValueError(f"Missing or duplicate values in {name}.")
pd.testing.assert_frame_equal(
    manifest[keys].sort_values("image_id").reset_index(drop=True),
    split_df[keys].sort_values("image_id").reset_index(drop=True),
    obj="Manifest and saved split; rerun notebook 01 if data changed"
)
if "split" not in split_df or split_df["split"].isna().any() or set(split_df["split"]) != {"train", "val"}:
    raise ValueError("Expected only train and val split assignments.")
class_names = sorted(manifest.label.unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
declared = pd.read_csv(DATA_DIR / "classes.csv", dtype=str).label
assert declared.is_unique and set(declared) == set(class_names)
train_df = split_df.loc[split_df["split"] == "train"].reset_index(drop=True)
val_df = split_df.loc[split_df["split"] == "val"].reset_index(drop=True)
assert set(train_df.image_id).isdisjoint(val_df.image_id)
assert set(train_df.label) == set(val_df.label) == set(class_names)
missing = [p for p in split_df.image_path if not (IMAGE_ROOT / p).is_file()]
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} images: {missing[:5]}")
split_sha256 = hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest()
print(f"Training: {len(train_df):,}; validation: {len(val_df):,}; classes: {len(class_names)}")

## 2. ImageNet preprocessing and pretrained model
Training uses random resized crops and horizontal flips. Validation uses deterministic resize and center crop. Both produce RGB tensors of shape 3 × 224 × 224 with ImageNet normalization. The competition test set remains untouched.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
eval_transform = weights.transforms()
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=eval_transform.mean, std=eval_transform.std)
])
train_dataset = Indoor67Dataset(train_df, IMAGE_ROOT, class_to_idx, train_transform)
val_dataset = Indoor67Dataset(val_df, IMAGE_ROOT, class_to_idx, eval_transform)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    pin_memory=DEVICE.type == "cuda", generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=DEVICE.type == "cuda"
)
model = models.resnet18(weights=weights)
feature_dim = model.fc.in_features
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(feature_dim, len(class_names))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
print("Weights:", weights.name, "| Features:", feature_dim, "| Outputs:", model.fc.out_features)

## 3. Cache frozen features
Keep the backbone in evaluation mode so its batch-normalization statistics stay fixed. Features and labels are kept together, preserving alignment even when training images are shuffled. No scaler or feature extractor is fitted on validation images.

In [ ]:
@torch.inference_mode()
def extract_backbone_features(loader, description):
    model.eval()
    features, labels = [], []
    for images, targets in tqdm(loader, desc=description):
        x = images.to(DEVICE)
        x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
        x = model.layer4(model.layer3(model.layer2(model.layer1(x))))
        features.append(torch.flatten(model.avgpool(x), 1).cpu())
        labels.append(targets)
    # Ordinary tensors are required for autograd when training the classifier.
    return torch.cat(features), torch.cat(labels)

start = time.perf_counter()
cached_train, cached_train_labels = extract_backbone_features(train_loader, "Training features")
cached_val, cached_val_labels = extract_backbone_features(val_loader, "Validation features")
# Clone outside inference_mode so these can be saved for backward by Linear.
cached_train, cached_val = cached_train.clone(), cached_val.clone()
assert cached_train.shape == (len(train_df), feature_dim)
assert cached_val.shape == (len(val_df), feature_dim)
assert torch.isfinite(cached_train).all() and torch.isfinite(cached_val).all()
assert torch.equal(cached_val_labels, torch.tensor(val_df.label.map(class_to_idx).to_numpy()))
feature_loader = DataLoader(
    TensorDataset(cached_train, cached_train_labels), batch_size=128, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)
print(f"Feature extraction: {time.perf_counter() - start:.1f}s")

## 4. Train the new classifier
Fit only the new classification layer. Save the best complete model across both training stages, choosing by validation accuracy and then macro F1 to break ties. History and checkpoint files are replaced by each new run.

In [ ]:
history = []
best_score = (-1.0, -1.0)
best_state = None

def record_epoch(stage, epoch, train_loss, train_accuracy, val_loss, targets, predictions, seconds):
    global best_score, best_state
    accuracy = accuracy_score(targets, predictions)
    macro_f1 = f1_score(targets, predictions, labels=range(len(class_names)), average="macro", zero_division=0)
    row = dict(stage=stage, epoch=epoch, train_loss=train_loss, train_accuracy=train_accuracy,
               val_loss=val_loss, val_accuracy=accuracy, val_macro_f1=macro_f1, seconds=seconds)
    history.append(row)
    if (accuracy, macro_f1) > best_score:
        best_score = (accuracy, macro_f1)
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        torch.save({
            "model_state_dict": best_state, "architecture": "resnet18",
            "class_names": class_names, "class_to_idx": class_to_idx,
            "weights": weights.name, "split_sha256": split_sha256, "seed": SEED,
            "best_epoch": row, "image_size": 224,
            "resize_size": list(eval_transform.resize_size),
            "mean": eval_transform.mean, "std": eval_transform.std,
            "head_epochs": HEAD_EPOCHS, "finetune_epochs": FINETUNE_EPOCHS,
            "head_lr": HEAD_LR, "finetune_lr": FINETUNE_LR,
            "weight_decay": WEIGHT_DECAY
        }, BEST_PATH)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print(f"{stage} {epoch}: train loss={train_loss:.4f}, val loss={val_loss:.4f}, "
          f"val accuracy={accuracy:.2%}, macro F1={macro_f1:.4f}, {seconds:.1f}s", flush=True)

optimizer = torch.optim.AdamW(model.fc.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
for epoch in range(1, HEAD_EPOCHS + 1):
    start = time.perf_counter()
    model.eval()
    model.fc.train()
    total_loss, correct, count = 0.0, 0, 0
    for features, targets in feature_loader:
        features, targets = features.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model.fc(features)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
        correct += (logits.argmax(1) == targets).sum().item()
        count += len(targets)
    model.eval()
    with torch.inference_mode():
        logits = model.fc(cached_val.to(DEVICE))
        val_loss = criterion(logits, cached_val_labels.to(DEVICE)).item()
        predictions = logits.argmax(1).cpu().numpy()
    record_epoch("head", epoch, total_loss / count, correct / count,
                 val_loss, cached_val_labels.numpy(), predictions, time.perf_counter() - start)

## 5. Fine-tune the final residual block
Start from the best classifier checkpoint. Unfreeze `layer4` and the classifier, use a smaller learning rate, and load fresh augmented images. Keep batch normalization in evaluation mode to retain ImageNet running statistics on this small dataset. Earlier blocks stay frozen.

In [ ]:
model.load_state_dict(best_state)
for name, parameter in model.named_parameters():
    parameter.requires_grad = name.startswith("layer4.") or name.startswith("fc.")
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY
)

@torch.inference_mode()
def evaluate(loader):
    model.eval()
    total_loss, count = 0.0, 0
    targets_all, predictions_all = [], []
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        total_loss += criterion(logits, targets).item() * len(targets)
        count += len(targets)
        targets_all.extend(targets.cpu().tolist())
        predictions_all.extend(logits.argmax(1).cpu().tolist())
    return total_loss / count, np.asarray(targets_all), np.asarray(predictions_all)

for epoch in range(1, FINETUNE_EPOCHS + 1):
    start = time.perf_counter()
    model.eval()
    model.layer4.train()
    model.fc.train()
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
    total_loss, correct, count = 0.0, 0, 0
    for images, targets in tqdm(train_loader, desc=f"Fine-tune {epoch}"):
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
        correct += (logits.argmax(1) == targets).sum().item()
        count += len(targets)
    val_loss, targets, predictions = evaluate(val_loader)
    record_epoch("finetune", epoch, total_loss / count, correct / count,
                 val_loss, targets, predictions, time.perf_counter() - start)

## 6. Evaluate the best checkpoint and compare with the baseline
Reload the saved checkpoint and evaluate deterministic validation images. Baseline comparison requires the same split fingerprint. Validation scores are model-selection estimates, not a held-out test result.

In [ ]:
checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=True)
assert checkpoint["class_to_idx"] == class_to_idx
assert checkpoint["split_sha256"] == split_sha256
model.load_state_dict(checkpoint["model_state_dict"])
val_loss, y_val, y_pred = evaluate(val_loader)
accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, labels=range(len(class_names)), average="macro", zero_division=0)
metrics = pd.DataFrame([dict(
    model="resnet18_transfer", accuracy=accuracy, macro_f1=macro_f1,
    seed=SEED, train_images=len(train_df), val_images=len(val_df),
    classes=len(class_names), split_sha256=split_sha256
)])
metrics.to_csv(EXPERIMENT_DIR / "resnet18_metrics.csv", index=False)
baseline_path = EXPERIMENT_DIR / "baseline_metrics.csv"
comparison = metrics
if baseline_path.is_file():
    baseline = pd.read_csv(baseline_path)
    if "split_sha256" not in baseline or not baseline.split_sha256.eq(split_sha256).all():
        raise ValueError("Rerun the baseline on this split before comparing results.")
    comparison = pd.concat([baseline, metrics], ignore_index=True)
display(comparison[["model", "accuracy", "macro_f1", "train_images", "val_images"]])
print("Best checkpoint:", checkpoint["best_epoch"])
report = pd.DataFrame(classification_report(
    y_val, y_pred, labels=range(len(class_names)), target_names=class_names,
    output_dict=True, zero_division=0
)).T
report.to_csv(EXPERIMENT_DIR / "resnet18_classification_report.csv")
predictions = val_df[keys].copy()
predictions["predicted_label"] = np.asarray(class_names)[y_pred]
predictions.to_csv(EXPERIMENT_DIR / "resnet18_validation_predictions.csv", index=False)
display(report)

## 7. Inspect learning curves and mistakes
The stage boundary changes the training objective from cached views to fresh augmentations, so training losses across that boundary are not directly equivalent. Validation preprocessing stays fixed.

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
steps = np.arange(1, len(history_df) + 1)
axes[0].plot(steps, history_df.train_loss, label="Train")
axes[0].plot(steps, history_df.val_loss, label="Validation")
axes[0].set(title="Loss", xlabel="Epoch")
axes[0].legend()
axes[1].plot(steps, history_df.val_accuracy, label="Accuracy")
axes[1].plot(steps, history_df.val_macro_f1, label="Macro F1")
axes[1].set(title="Validation scores", xlabel="Epoch")
axes[1].legend()
for ax in axes:
    ax.axvline(HEAD_EPOCHS + 0.5, color="gray", linestyle="--")
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_val, y_pred, labels=range(len(class_names)))
fig, ax = plt.subplots(figsize=(16, 14))
plot = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(len(class_names)), yticks=range(len(class_names)),
       xticklabels=class_names, yticklabels=class_names,
       xlabel="Predicted class", ylabel="True class", title="ResNet18 validation confusion matrix")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.setp(ax.get_yticklabels(), fontsize=7)
fig.colorbar(plot, ax=ax)
plt.tight_layout()
plt.show()

from PIL import Image
errors = predictions.loc[predictions.label != predictions.predicted_label]
if len(errors):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, errors.sample(min(8, len(errors)), random_state=SEED).itertuples()):
        with Image.open(IMAGE_ROOT / row.image_path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"True: {row.label}\nPredicted: {row.predicted_label}", fontsize=9)
    plt.tight_layout()
    plt.show()
print("Best model:", BEST_PATH)
print("Metrics and predictions:", EXPERIMENT_DIR)

## Next experiment
Compare the scores and mistakes above with notebook 02. Consider a larger fine-tuning budget or a different pretrained architecture while keeping this split fixed. Generate competition test predictions only after choosing the final model; this notebook does not create or submit a competition entry.